In [1]:
import rasterio
import numpy as np
import os


In [2]:
import rasterio
import numpy as np

tile_size = 512

with rasterio.open("KNL_DDP_cotton crop.jpg") as src:

    img = src.read()   # shape = (bands, height, width)

    height = img.shape[1]
    width = img.shape[2]

    pad_h = (tile_size - height % tile_size) % tile_size
    pad_w = (tile_size - width % tile_size) % tile_size

    padded = np.pad(
        img,
        ((0,0),(0,pad_h),(0,pad_w)),
        mode="constant",
        constant_values=0
    )
    profile = src.profile
    profile.update(
    height=padded.shape[1],
    width=padded.shape[2])
    with rasterio.open("KNL_DDP_cotton crop_new.jpg", "w", **profile) as dst:
        dst.write(padded)

print("Original:", img.shape)
print("Padded:", padded.shape)




C:\Users\CSE_SDPL\anaconda3\envs\drought\lib\site-packages\rasterio\__init__.py:368: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)
C:\Users\CSE_SDPL\anaconda3\envs\drought\lib\site-packages\rasterio\__init__.py:378: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(


Original: (3, 49690, 22189)
Padded: (3, 50176, 22528)


In [3]:
import rasterio
import numpy as np
import os

tile_size = 512
output_folder = "dataset/images"

os.makedirs(output_folder, exist_ok=True)

with rasterio.open("KNL_DDP_cotton crop_new.jpg") as src:

    img = src.read([1,2,3])
    img = np.transpose(img, (1,2,0))

    h, w, _ = img.shape

    count = 0

    for y in range(0, h, tile_size):
        for x in range(0, w, tile_size):

            tile = img[y:y+tile_size, x:x+tile_size]

            if tile.shape[0] == tile_size and tile.shape[1] == tile_size:

                filename = f"{output_folder}/tile_{count}.png"

                import cv2
                cv2.imwrite(filename, tile)

                count += 1

print("Tiles created:", count)

Tiles created: 4312


In [4]:
#mask generation
import os
import cv2
import numpy as np
from tqdm import tqdm

image_folder = "dataset/images"
mask_folder = "dataset/masks"

os.makedirs(mask_folder, exist_ok=True)

files = os.listdir(image_folder)

for file in tqdm(files):

    path = os.path.join(image_folder, file)

    img = cv2.imread(path)


    img = img.astype(np.float32)

    B = img[:,:,0]
    G = img[:,:,1]
    R = img[:,:,2]

    # Excess Green Index
    exg = 2*G - R - B

    # Normalize
    exg = cv2.normalize(exg, None, 0, 255, cv2.NORM_MINMAX)

    exg = exg.astype(np.uint8)

    # Threshold
    _, mask = cv2.threshold(exg, 110, 255, cv2.THRESH_BINARY_INV)

    # Remove noise
    kernel = np.ones((5,5),np.uint8)

    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    save_path = os.path.join(mask_folder, file)

    cv2.imwrite(save_path, mask)

print("Masks generated")

100%|████████████████████████████████████████████████████████████| 4312/4312 [01:51<00:00, 38.52it/s]

Masks generated


In [5]:
import os
import cv2
import torch
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm

########################################
# Dataset Loader
########################################

class DroneDataset(Dataset):

    def __init__(self, img_dir, mask_dir):

        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = os.listdir(img_dir)

        self.transform = transforms.Compose([
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img_path = os.path.join(self.img_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.images[idx])

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(mask_path, 0)

        image = image / 255.0
        mask = mask / 255.0

        image = np.transpose(image, (2,0,1))
        mask = np.expand_dims(mask, axis=0)

        return torch.tensor(image).float(), torch.tensor(mask).float()


########################################
# U-Net Model
########################################

class DoubleConv(nn.Module):

    def __init__(self, in_c, out_c):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.ReLU()
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.down1 = DoubleConv(3, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(64,128)
        self.pool2 = nn.MaxPool2d(2)

        self.middle = DoubleConv(128,256)

        self.up1 = nn.ConvTranspose2d(256,128,2,2)
        self.conv1 = DoubleConv(256,128)

        self.up2 = nn.ConvTranspose2d(128,64,2,2)
        self.conv2 = DoubleConv(128,64)

        self.final = nn.Conv2d(64,1,1)

    def forward(self,x):

        d1 = self.down1(x)
        p1 = self.pool1(d1)

        d2 = self.down2(p1)
        p2 = self.pool2(d2)

        m = self.middle(p2)

        u1 = self.up1(m)
        c1 = torch.cat([u1,d2],dim=1)
        c1 = self.conv1(c1)

        u2 = self.up2(c1)
        c2 = torch.cat([u2,d1],dim=1)
        c2 = self.conv2(c2)

        out = self.final(c2)

        return torch.sigmoid(out)


########################################
# Training Setup
########################################

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = DroneDataset(
    "dataset/images",
    "dataset/masks"
)

loader = DataLoader(dataset, batch_size=4, shuffle=True)

model = UNet().to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

epochs = 1


########################################
# Training Loop
########################################

for epoch in range(epochs):

    model.train()
    total_loss = 0

    loop = tqdm(loader)

    for images, masks in loop:

        images = images.to(device)
        masks = masks.to(device)

        preds = model(images)

        loss = criterion(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        loop.set_description(f"Epoch {epoch+1}/{epochs}")
        loop.set_postfix(loss=loss.item())

    print("Epoch Loss:", total_loss/len(loader))
    os.makedirs("models", exist_ok=True)
    torch.save(model.state_dict(), "models/unet_drought.pth")


########################################
# Save Model
########################################

os.makedirs("models", exist_ok=True)

torch.save(model.state_dict(), "models/unet_drought_final.pth")

print("Model saved")

Epoch 1/1: 100%|████████████████████████████████████| 1078/1078 [08:47<00:00,  2.04it/s, loss=0.0458]

Epoch Loss: 0.12812398785936308
Model saved


In [6]:
import os
import cv2
import torch
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm

########################################
# Dataset Loader
########################################

class DroneDataset(Dataset):

    def __init__(self, img_dir, mask_dir):

        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = os.listdir(img_dir)

        self.transform = transforms.Compose([
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img_path = os.path.join(self.img_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.images[idx])

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(mask_path, 0)

        image = image / 255.0
        mask = mask / 255.0

        image = np.transpose(image, (2,0,1))
        mask = np.expand_dims(mask, axis=0)

        return torch.tensor(image).float(), torch.tensor(mask).float()


########################################
# U-Net Model
########################################

class DoubleConv(nn.Module):

    def __init__(self, in_c, out_c):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.ReLU()
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.down1 = DoubleConv(3, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(64,128)
        self.pool2 = nn.MaxPool2d(2)

        self.middle = DoubleConv(128,256)

        self.up1 = nn.ConvTranspose2d(256,128,2,2)
        self.conv1 = DoubleConv(256,128)

        self.up2 = nn.ConvTranspose2d(128,64,2,2)
        self.conv2 = DoubleConv(128,64)

        self.final = nn.Conv2d(64,1,1)

    def forward(self,x):

        d1 = self.down1(x)
        p1 = self.pool1(d1)

        d2 = self.down2(p1)
        p2 = self.pool2(d2)

        m = self.middle(p2)

        u1 = self.up1(m)
        c1 = torch.cat([u1,d2],dim=1)
        c1 = self.conv1(c1)

        u2 = self.up2(c1)
        c2 = torch.cat([u2,d1],dim=1)
        c2 = self.conv2(c2)

        out = self.final(c2)

        return torch.sigmoid(out)

In [7]:
import torch
import cv2
import numpy as np
import os

model = UNet()
model.load_state_dict(torch.load("models/unet_drought.pth"))
model.eval()

tile_folder = "dataset/images"
out_folder = "dataset/predictions"

os.makedirs(out_folder, exist_ok=True)

for file in os.listdir(tile_folder):

    img = cv2.imread(f"{tile_folder}/{file}")
    img = cv2.resize(img,(512,512))

    img = img.transpose(2,0,1)/255.0
    img = torch.tensor(img).float().unsqueeze(0)

    with torch.no_grad():
        pred = model(img)

    mask = pred.squeeze().numpy()

    mask = (mask>0.5)*255

    cv2.imwrite(f"{out_folder}/{file}", mask.astype("uint8"))

C:\Users\CSE_SDPL\AppData\Local\Temp\ipykernel_24268\585455079.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("models/unet_drought.pth"

In [12]:
import cv2
import numpy as np
import os
import re
tile_size = 512

rows = 114
cols = 48

full = np.zeros((rows*tile_size, cols*tile_size))
tiles1 = os.listdir("dataset/predictions")
tiles = sorted(tiles1, key=lambda x: int(re.search(r'tile_(\d+)\.png', x).group(1)))

count = 0

for r in range(rows):
    for c in range(cols):

        tile = cv2.imread(f"dataset/predictions/{tiles[count]}",0)

        full[r*tile_size:(r+1)*tile_size,
             c*tile_size:(c+1)*tile_size] = tile

        count+=1

cv2.imwrite("drought_map_kurnool.png", full)

MemoryError: Unable to allocate 10.7 GiB for an array with shape (58368, 24576) and data type float64

In [13]:
import cv2
import numpy as np
import os
import re
import rasterio

# ============================================================
INPUT_IMAGE        = "KNL_DDP_cotton crop_new.jpg"  # padded image
PREDICTIONS_FOLDER = "dataset/predictions"
OUTPUT_MAP         = "drought_map_kurnool.png"
# ============================================================

tile_size = 512

# --- Auto-compute everything from image ---
with rasterio.open(INPUT_IMAGE) as src:
    original_h = src.height
    original_w = src.width

pad_h    = (tile_size - original_h % tile_size) % tile_size
pad_w    = (tile_size - original_w % tile_size) % tile_size
padded_h = original_h + pad_h
padded_w = original_w + pad_w

rows = padded_h // tile_size
cols = padded_w // tile_size

print(f"Original : {original_h} x {original_w}")
print(f"Padded   : {padded_h} x {padded_w}")
print(f"Grid     : {rows} rows x {cols} cols = {rows*cols} tiles expected")

# --- Sort tiles ---
tiles = sorted(
    os.listdir(PREDICTIONS_FOLDER),
    key=lambda x: int(re.search(r'tile_(\d+)\.png', x).group(1))
)
print(f"Tiles found: {len(tiles)}")

# --- Stitch — uint8 saves 8x memory vs float64 ---
full = np.zeros((padded_h, padded_w), dtype=np.uint8)  # ✅ fix: uint8 not float64

count = 0
for r in range(rows):
    for c in range(cols):
        if count >= len(tiles):
            break
        tile = cv2.imread(f"{PREDICTIONS_FOLDER}/{tiles[count]}", 0)
        if tile is not None:
            full[r*tile_size:(r+1)*tile_size,
                 c*tile_size:(c+1)*tile_size] = tile
        count += 1

# --- Crop padding ---
full_cropped = full[:original_h, :original_w]

cv2.imwrite(OUTPUT_MAP, full_cropped)
print(f"✅ Saved: {OUTPUT_MAP}  |  Size: {full_cropped.shape}")

Original : 50176 x 22528
Padded   : 50176 x 22528
Grid     : 98 rows x 44 cols = 4312 tiles expected
Tiles found: 4312
✅ Saved: drought_map_kurnool.png  |  Size: (50176, 22528)


In [13]:
import rasterio
import numpy as np

with rasterio.open("drought_map_somandevpalli.png") as src:
    
    mask = src.read(1)

veg_pixels = np.sum(mask == 0)
total_pixels = mask.size

veg_percentage = (veg_pixels / total_pixels) * 100

print("Vegetation pixels:", veg_pixels)
print("Total pixels:", total_pixels)
print("Vegetation %:", veg_percentage)

E:\ProgramData\anaconda3\envs\tf\lib\site-packages\rasterio\__init__.py:387: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


Vegetation pixels: 39777017
Total pixels: 866648064
Vegetation %: 4.589754325003604


In [15]:
# remove white images
import cv2
import os
import numpy as np

folder_path = "dataset/images"   # change to your folder
cnt=0
for file in os.listdir(folder_path):
    if file.lower().endswith((".png", ".jpg", ".jpeg")):
        path = os.path.join(folder_path, file)

        img = cv2.imread(path)

        if img is None:
            continue

        # Check size
        if img.shape[0] == 512 and img.shape[1] == 512:
            # Check if image is completely white
            if np.all(img == 255):
                cnt=cnt+1
print(cnt)

362
